# 📈 Exploratory Data Analysis & Strategic Insights### Retail Business Intelligence & Customer Analytics Platform**Author:** Senior Data Analyst & BI Developer**Objective:** Uncover revenue trends, category profitability drivers, regional disparities, discount erosion effects, and operational delivery bottlenecks.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Styling configuration
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.autolayout'] = True
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.labelsize'] = 12

df = pd.read_csv('../data/processed/superstore_enriched.csv')
df['order_date'] = pd.to_datetime(df['order_date'])
print(f"Enriched dataset loaded: {df.shape[0]:,} records")

## 1. Executive Summary KPIs

In [ ]:
total_sales = df['sales'].sum()
total_profit = df['profit'].sum()
profit_margin = (total_profit / total_sales) * 100
total_orders = df['order_id'].nunique()
total_customers = df['customer_id'].nunique()
aov = total_sales / total_orders

print("========================================")
print("     EXECUTIVE KPI SCORECARD")
print("========================================")
print(f"Total Revenue:         ${total_sales:,.2f}")
print(f"Total Net Profit:      ${total_profit:,.2f}")
print(f"Overall Profit Margin:  {profit_margin:.2f}%")
print(f"Total Orders Placed:    {total_orders:,}")
print(f"Total Active Customers: {total_customers:,}")
print(f"Average Order Value:   ${aov:.2f}")
print("========================================")

## 2. Monthly Revenue & Profitability Trajectory

In [ ]:
monthly = df.groupby('year_month').agg(
    monthly_sales=('sales', 'sum'),
    monthly_profit=('profit', 'sum')
).reset_index()

fig, ax1 = plt.subplots(figsize=(14, 5))

color = '#1f77b4'
ax1.set_xlabel('Year-Month')
ax1.set_ylabel('Sales ($)', color=color)
ax1.plot(monthly['year_month'], monthly['monthly_sales'], color=color, marker='o', linewidth=2.5, label='Sales')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_xticks(range(0, len(monthly), 3))
ax1.set_xticklabels(monthly['year_month'].iloc[::3], rotation=45)

ax2 = ax1.twinx()
color = '#2ca02c'
ax2.set_ylabel('Profit ($)', color=color)
ax2.plot(monthly['year_month'], monthly['monthly_profit'], color=color, marker='s', linestyle='--', linewidth=2, label='Profit')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Monthly Sales and Net Profit Growth (2014 - 2017)')
plt.show()

## 3. Category & Sub-Category Performance Analysis

In [ ]:
cat_summary = df.groupby(['category', 'sub_category']).agg(
    sales=('sales', 'sum'),
    profit=('profit', 'sum')
).reset_index().sort_values('profit', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Sales by Sub-Category
sns.barplot(data=cat_summary, y='sub_category', x='sales', hue='category', ax=axes[0], dodge=False)
axes[0].set_title('Total Revenue by Sub-Category')
axes[0].set_xlabel('Sales ($)')
axes[0].set_ylabel('Sub-Category')

# Profit by Sub-Category
colors = ['#d62728' if x < 0 else '#2ca02c' for x in cat_summary['profit']]
axes[1].barh(cat_summary['sub_category'], cat_summary['profit'], color=colors)
axes[1].set_title('Net Profit / Loss by Sub-Category')
axes[1].set_xlabel('Profit ($)')
axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--')

plt.tight_layout()
plt.show()

## 4. The Profit Erosion Trap: Discount vs Profit Margin

In [ ]:
plt.figure(figsize=(10, 5))
discount_analysis = df.groupby('discount_band').agg(
    total_sales=('sales', 'sum'),
    total_profit=('profit', 'sum'),
    avg_margin=('profit_margin', lambda x: (df.loc[x.index, 'profit'].sum() / df.loc[x.index, 'sales'].sum()) * 100)
).reset_index()

sns.barplot(data=discount_analysis, x='discount_band', y='avg_margin', palette='RdYlGn_r')
plt.axhline(0, color='red', linestyle='--', linewidth=1.2)
plt.title('Profit Margin (%) by Discount Band: Severe Margin Destruction Above 20%')
plt.xlabel('Discount Band')
plt.ylabel('Effective Profit Margin (%)')
plt.show()

display(discount_analysis)

## 5. Key EDA Findings & Strategic Takeaways1. **Seasonal Q4 Peaks**: Strong recurring sales surges in November and December of every year.2. **Loss-Bleeding Subcategories**: **Tables (-$17,725.48)** and **Bookcases (-$3,472.56)** are the primary sources of margin loss.3. **Dangerous Discount Threshold**: While discounts up to 20% maintain positive margins (10-15%), any discount > 20% generates steep losses (-30% to -80% margins).4. **Regional Dominance**: West and East regions deliver 65%+ of company profits, while the Central region suffers from aggressive discounting in furniture.